In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

from langchain_core.prompts import PromptTemplate

modelo_de_prompt = PromptTemplate(
    template="Explica de manera clara, concreta, practica y actualizada, cuáles son las 3 estrategias sicologicas en el área de {tema}.",
    input_variables=["tema"],
)

from langchain_core.output_parsers import StrOutputParser

#Aqui se esta invocando directamente el LLM con un promptTemplate
cadena = modelo_de_prompt | llm | StrOutputParser()

respuesta = cadena.invoke({"tema": "ventas de paginas web"})

print(respuesta)


In [5]:

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool
import os
  #"""Realiza una búsqueda en la web sobre un tema específico"""  -- Con este el agente o llm lo llaman, solo que llm no lo ejecuta
  #"""Realiza una búsqueda exclusivamente sobre fútbol. No debe utilizarse para ningún otro tema.""" -- Con este agente ni llm lo usan
@tool
def busca_web(query: str) -> list:
  """Realiza una búsqueda en la web sobre un tema específico"""
  tavily_search = TavilySearchResults(
      max_results=2,
      search_depth="advanced",
      max_tokens=1000
  )
  resultado_busca = tavily_search.invoke(query)
  return resultado_busca

#Aqui estamos usando una tool de python para ejecutar Tavily
#busca_web.invoke("Impacto de la IA en las ventas de servicios")


C:\Users\tcpad\AppData\Local\Temp\ipykernel_24412\2244567245.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [8]:

tools = [busca_web]

#Aqui estamos agregando la funcion como herramienta
llm_con_herramienta = llm.bind(tools=tools)

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

modelo_de_prompt = PromptTemplate(
    template=" Usando solo las herramientas disponibles, dime cuáles son los 2 impactos de la inteligencia artificial en el área de {tema}.",
    input_variables=["tema"],
)

# Aqui no se esta enviando el tema
# prompt = "Usando solo las tools disponibles, dime cuáles son los impactos de la inteligencia artificial en el área de {tema}."
# respuesta = llm_con_herramienta.invoke(prompt)
# print(respuesta)

#OJO Este llamado pide la herramienta pero nunca la ejecuta, si quito sin ellas no hace nada
cadena = modelo_de_prompt | llm_con_herramienta | StrOutputParser()
#respuesta =  cadena.invoke({"tema": "dinero"})
print(respuesta)
# print(respuesta.tool_calls) Muestra las herramientas asociadas

Para vender páginas web de manera efectiva, no basta con hablar de código o diseño. Debes vender **resultados de negocio**. Aquí tienes las 3 estrategias psicológicas más potentes y actualizadas para cerrar ventas en este sector:

---

### 1. La Estrategia del "Costo de Oportunidad" (Evitar la Pérdida)
La psicología humana teme más a perder algo que a ganar algo nuevo (aversión a la pérdida). En lugar de vender "una web bonita", vende lo que están dejando de ganar por tener una web obsoleta.

*   **Cómo aplicarlo:** No digas: *"Tu nueva web será moderna"*.
*   **Di esto:** *"Tu web actual está haciendo que pierdas clientes cada día. Cada persona que entra y no encuentra lo que busca en 3 segundos se va a tu competencia. Tu web actual no es una herramienta, es un agujero negro por donde se te escapan ventas que ya estás pagando en publicidad"*.
*   **En la práctica:** Haz una auditoría gratuita rápida frente a ellos. Muéstrales los errores técnicos o de experiencia de usuario que les im

In [ ]:
#respuesta = (modelo_de_prompt | llm_con_herramienta).invoke({"tema": "educación"})
print(respuesta)
print(respuesta.tool_calls)

AttributeError: 'str' object has no attribute 'tool_calls'

In [9]:

from langgraph.prebuilt import create_react_agent

system_prompt = """
Actúa como un asistente útil y especializado en investigación.


Siempre que el usuario pregunte sobre un tema específico:

2. Analiza los resultados.
3. Devuelve una respuesta clara muy concreta.
4. Incluye los enlaces de las fuentes utilizadas.
"""

agente_web = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

resultado = ""

# resultado = agente_web.invoke({
#     "messages": [
#         ("user", "Cuáles son los impactos del consumo de sustancias en las personas que quieren emprender luego de 3 años de abstencion?")
#     ]
# })

#resultado["messages"][-1]


C:\Users\tcpad\AppData\Local\Temp\ipykernel_24412\4035261324.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_web = create_react_agent(


In [10]:
from langchain_core.messages import ToolMessage

#Comprueba si uso la herramienta
uso_tool = any(isinstance(msg, ToolMessage) for msg in resultado["messages"])
print(uso_tool)

#Herramientas usadas en el proceso
for msg in resultado["messages"]:
    print(type(msg).__name__)

#Seguimiento de que hace cada elemento de langchain
for msg in resultado["messages"]:
    print(type(msg).__name__)
    print("content:", msg.content)
    print("tool_calls:", getattr(msg, "tool_calls", None))
    print("------")

TypeError: string indices must be integers, not 'str'

In [11]:
import arxiv

@tool
def buscar_arxiv(query: str) -> str:
    """Busca artículos científicos en arXiv y devuelve títulos y enlaces."""
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=3,
        sort_by=arxiv.SortCriterion.Relevance
    )

    resultados = []
    for paper in client.results(search):
        resultados.append(
            f"Título: {paper.title}\nURL: {paper.entry_id}\n"
        )
    return "\n".join(resultados)

system_prompt2 = """
Actúa como un asistente útil.
Devuelve los títulos encontrados.
"""

tools_cientifica = [buscar_arxiv]
agente_cientifico = create_react_agent(
    model=llm,
    tools=tools_cientifica,
    prompt=system_prompt2
)

resultado = agente_cientifico.invoke({
    "messages": [
        ("user", "multi-agent systems")
    ]
})

respuesta_final = resultado["messages"][-1].content

# Si es una lista, imprime el texto del primer bloque. Si ya es un string, lo imprime directamente.
if isinstance(respuesta_final, list):
    print(respuesta_final[0]["text"])
else:
    print(respuesta_final)

C:\Users\tcpad\AppData\Local\Temp\ipykernel_24412\1495981393.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_cientifico = create_react_agent(


Aquí tienes algunos títulos de artículos científicos recientes sobre Sistemas Multi-agente encontrados en arXiv:

1. **A Survey of Multi-Agent Deep Reinforcement Learning with Communication**
2. **A Methodology to Engineer and Validate Dynamic Multi-level Multi-agent Based Simulations**
3. **Augmenting the action space with conventions to improve multi-agent cooperation in Hanabi**


In [12]:

from typing import TypedDict

class AgentState(TypedDict):
    user_query: str
    web_answer: str
    scientific_answer: str
    final_answer: str

def funcion_agente_web(state: AgentState) -> dict:
    """
    Función que ejecuta un agente al estilo ReAct que realiza una búsqueda en la web
    sobre un término específico.
    """

    system_prompt = """
    Actúa como un asistente útil.
    Usa las herramientas proporcionadas para responder a las preguntas del usuario.

    - busca_web: Retorna resultados de búsqueda web.

    Cuando el usuario pregunte sobre un tema específico, usa busca_web y devuelve enlaces en la respuesta.
    """

    agente_web = create_react_agent(
        model=llm,
        tools=tools,
        prompt=system_prompt
    )

    resultado = agente_web.invoke({
        "messages": [("user", state["user_query"])]
    })

    respuesta_final = resultado["messages"][-1].text
    return {"web_answer": respuesta_final}

def funcion_agente_cientifico(state: AgentState) -> dict:
    """
    Función que ejecuta un agente al estilo ReAct que realiza una búsqueda en arXiv
    sobre un término específico.
    """

    system_prompt = """
    Actúa como un asistente útil.
    Usa las herramientas proporcionadas para responder a las preguntas del usuario.

    - tool_cientifica: Retorna resultados de una búsqueda en arXiv.

    Cuando el usuario pregunte sobre un tema específico, usa tool_cientifica y devuelve los títulos de los artículos.
    """

    agente_cientifico = create_react_agent(
        model=llm,
        tools=tools_cientifica,
        prompt=system_prompt
    )

    resultado = agente_cientifico.invoke({
        "messages": [("user", state["user_query"])]
    })

    respuesta_final = resultado["messages"][-1].text
    return {"scientific_answer": respuesta_final}


In [13]:
#Definir el grafo para combinar agentes
from langgraph.graph import START, StateGraph, END

workflow = StateGraph(AgentState)

workflow.add_node("agente_web", funcion_agente_web)
workflow.add_edge(START, "agente_web")
workflow.add_edge("agente_web", END)

app = workflow.compile()
#resultado = app.invoke({"user_query": "Cuál es el impacto de la IA en la medicina"})
#print(resultado["web_answer"])


# from IPython.display import Image, display
# Image(app.get_graph().draw_mermaid_png())


In [14]:
def supervisor_node(state: AgentState) -> dict:
  """
  Combina los resultados de los agentes web y cientificos y formatea la respuesta final.
  Args:
      state (AgentState): El estado actual del agente, que contiene web_answer y scientific_answer.
  Returns:
      dict: Un diccionario que contiene la respuesta final combinada y formateada.
  """

  web_results = state.get("web_answer", "No se realizó búsqueda web.")
  scientific_results = state.get("scientific_answer", "No se realizó búsqueda científica.")
  final_answer = '## Resultados combinados de búsqueda\n\n'
  final_answer += f"### Resultados de búsqueda web:\n\n" + web_results + "\n\n"
  final_answer += f"### Artículos científicos\n\n" + scientific_results + "\n\n"

  return {"final_answer": final_answer}

workflow = StateGraph(AgentState)
workflow.add_node("web_search", funcion_agente_web)
workflow.add_node("scientific_search", funcion_agente_cientifico)
workflow.add_node("supervisor", supervisor_node)


workflow.add_edge(START, "web_search")
workflow.add_edge(START, "scientific_search")
workflow.add_edge("web_search", "supervisor")
workflow.add_edge("scientific_search", "supervisor")
workflow.add_edge("supervisor", END)

app = workflow.compile()

#resultado = app.invoke({"user_query": "Por que personas con IQ elevado fracasan?"})
#print(resultado["final_answer"])

# Image(app.get_graph().draw_mermaid_png())


In [15]:
def router_agent(state: AgentState) -> dict:
  router_prompt = """
  Eres un agente ruteador. Tu tarea es decidir cuál agente es más apropiado
  para responder la pregunta del usuario.

  Pregunta del usuario: {user_query}

  Puedes elegir entre las siguientes opciones:
  - web_search: Usa esta opción si la pregunta requiere una búsqueda general en internet.
  - scientific_search: Usa esta opción si la pregunta requiere artículos científicos.

  Responde ÚNICAMENTE con el nombre de la opción (web_search o scientific_search).
  """

  router_agent_instance = create_react_agent(
  model=llm,
  tools=[],
  prompt=router_prompt
  )

  response = router_agent_instance.invoke({
    "messages": [("user", state["user_query"])]
  })

  decision = response["messages"][-1].text.strip().lower()
  if decision == "web_search":
      return {"router_decision": "web_search"}
  elif decision == "scientific_search":
      return {"router_decision": "scientific_search"}
  else:
      # Por defecto, usamos web_search
      return {"router_decision": "web_search"}

workflow = StateGraph(AgentState)
workflow.add_node("router", router_agent)
workflow.add_node("web_search", funcion_agente_web)
workflow.add_node("scientific_search", funcion_agente_cientifico)
workflow.add_node("supervisor", supervisor_node)

workflow.add_edge(START, "router")
workflow.add_conditional_edges(
    "router",
    lambda state: state["router_decision"],
    {
        "web_search": "web_search",
        "scientific_search": "scientific_search"
    }
)

workflow.add_edge("web_search", "supervisor")
workflow.add_edge("scientific_search", "supervisor")
workflow.add_edge("supervisor", END)
app = workflow.compile()

#resultado = app.invoke({"user_query": "Cuales son las 2 claves de los negocios de exito de servicios tecnologicos"})
#print(resultado["final_answer"])
# Image(app.get_graph().draw_mermaid_png())


In [17]:
import gradio as gr
def run_graph(user_query):
    """Ejecuta el LangGraph con la pregunta del usuario y devuelve la respuesta final."""
    result = app.invoke({"user_query": user_query})
    return result["final_answer"]

iface = gr.Interface(
    fn=run_graph,
    inputs=gr.Textbox(label="Escribe tu pregunta:"),
    outputs=gr.Markdown(label="Respuesta Final"),
    title="Agente de Búsqueda Inteligente",
    description="Haz una pregunta y obtén una respuesta usando fuentes web o científicas."
)

iface.launch()


d:\Proyectos\alianza-f1-knowledge-agent\Cursos\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


C:\Users\tcpad\AppData\Local\Temp\ipykernel_24412\1031837235.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  router_agent_instance = create_react_agent(
C:\Users\tcpad\AppData\Local\Temp\ipykernel_24412\3308713365.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_web = create_react_agent(
C:\Users\tcpad\AppData\Local\Temp\ipykernel_24412\2244567245.py:9: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from